# 8. GABAergic reclustering and lineage annotation

To focus on GABAergic interneuron lineages, we subset the integrated dataset to GABAergic cells and perform a dedicated reclustering and annotation. In this notebook, we:

- subset GABAergic cells based on canonical markers,
- recompute PCA, neighbors, and UMAP on the GABA subset,
- run Leiden clustering at multiple resolutions and choose a working resolution using silhouette scores,
- score cell cycle and lineage-specific gene sets (MGE, CGE, LGE progenitors and neuroblasts),
- assign lineage labels to clusters based on marker expression and differential expression, and
- quantify genotype and batch composition per GABA cluster.

The output is a GABA-focused AnnData object with cluster and lineage annotations and compositional summaries suitable for downstream figures.

## 8.1. Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata
from sklearn.metrics import silhouette_score

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sns.set(style="whitegrid")

SEED = 123
np.random.seed(SEED)

## 8.2. Load clustered dataset and subset GABAergic cells

We load the annotated, clustered dataset and subset to cells expressing GABAergic markers (Gad1/2, Dlx1/2). This defines the GABAergic subset to recluster.

In [ ]:
adata = anndata.read_h5ad("annotated.h5ad")
print(f"Full dataset: {adata.n_obs} cells × {adata.n_vars} genes")

# Define GABA markers and keep those present
gaba_marker_genes = ["Gad2", "Gad1", "Dlx1", "Dlx2"]
gaba_marker_genes = [g for g in gaba_marker_genes if g in adata.var_names]

X_gaba = adata[:, gaba_marker_genes].X
if hasattr(X_gaba, "A1"):  # sparse
    mask = (X_gaba > 0).sum(axis=1).A1 > 0
else:
    mask = (X_gaba > 0).sum(axis=1) > 0

adata_gaba = adata[mask].copy()
print(f"GABAergic subset: {adata_gaba.n_obs} cells × {adata_gaba.n_vars} genes")

## 8.3. PCA, neighbors, UMAP, and reclustering

We rescale the GABA subset, compute PCA, construct a neighbor graph, and then explore Leiden clustering at multiple resolutions. We also compute silhouette scores to help select a reasonable resolution. Here we choose a resolution that provides good separation while keeping clusters interpretable (e.g. 0.7).

In [ ]:
# Scale and PCA
sc.pp.scale(adata_gaba, max_value=10, zero_center=False)
sc.pp.pca(adata_gaba, n_comps=50, svd_solver="arpack")
sc.pp.neighbors(adata_gaba, use_rep="X_pca", n_neighbors=15)

# Store raw/lognorm counts before any further transforms
adata_gaba.layers["counts"] = adata_gaba.X.copy()

# Normalize and log1p for expression visualization
sc.pp.normalize_total(adata_gaba, target_sum=1e4)
sc.pp.log1p(adata_gaba)
adata_gaba.layers["log1p_norm_gaba"] = adata_gaba.X.copy()

# Clean any NaN/inf
adata_gaba.X = np.nan_to_num(adata_gaba.X, nan=0.0, posinf=10.0, neginf=-10.0)
print("NaNs and infinite values cleaned.")

# Leiden at a few fixed resolutions (for later use)
for res, key in [(0.6, "leiden_res0_6"),
                 (0.7, "leiden_res0_7"),
                 (0.8, "leiden_res0_8"),
                 (0.9, "leiden_res0_9"),
                 (1.5, "leiden_res1_5")]:
    sc.tl.leiden(adata_gaba, resolution=res, key_added=key)

# Systematic Leiden + silhouette scan
resolutions = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0]
sil_scores = {}
leiden_keys = [f"leiden_{res}" for res in resolutions]

for res, key in zip(resolutions, leiden_keys):
    sc.tl.leiden(adata_gaba, resolution=res, key_added=key)
    labels = adata_gaba.obs[key].astype(int)
    sil = silhouette_score(adata_gaba.obsm["X_pca"], labels)
    sil_scores[res] = sil
    print(f"Resolution {res:.2f} – silhouette score: {sil:.3f}")

# Compute UMAP
sc.tl.umap(adata_gaba, min_dist=0.3, spread=1.0)

In [ ]:
key_for_umap = "leiden_0.7"  # example; adjust to your choice
print(f"Using clustering key for annotation: {key_for_umap}")

## 8.4. Cell cycle scoring

We score S-phase and G2/M-phase gene sets where possible and derive a coarse cell cycle phase annotation (G1, S, G2/M). This helps to identify proliferation-enriched clusters (e.g. progenitors).

In [ ]:
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UNG", "GINS2", "MCM6",
    "CDCA7", "DTL", "PRIM1", "UHRF1", "MLF1IP", "HELLS", "RFC2", "RPA2", "NASP",
    "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2",
    "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8",
]

g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", "CKS2",
    "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "FAM64A", "SMC4", "CCNB2",
    "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1",
    "KIF20B", "HJURP", "CDCA3", "HN1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1",
    "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1",
    "ANLN", "LBR", "CKAP5", "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA",
]

s_genes_title = [g.title() for g in s_genes]
g2m_genes_title = [g.title() for g in g2m_genes]

s_genes_filtered = [g for g in s_genes_title if g in adata_gaba.var_names]
g2m_genes_filtered = [g for g in g2m_genes_title if g in adata_gaba.var_names]

print("Filtered S genes:", s_genes_filtered)
print("Filtered G2M genes:", g2m_genes_filtered)

if s_genes_filtered and g2m_genes_filtered:
    sc.tl.score_genes_cell_cycle(
        adata_gaba,
        s_genes=s_genes_filtered,
        g2m_genes=g2m_genes_filtered,
    )
    sc.pl.umap(
        adata_gaba,
        color=["S_score", "G2M_score"],
        cmap="viridis",
        title="UMAP: cell cycle scores",
    )

    adata_gaba.obs["cell_cycle_phase"] = np.where(
        adata_gaba.obs["S_score"] > adata_gaba.obs["G2M_score"],
        "S",
        np.where(
            adata_gaba.obs["G2M_score"] > adata_gaba.obs["S_score"],
            "G2M",
            "G1",
        ),
    )
    sc.pl.umap(
        adata_gaba,
        color="cell_cycle_phase",
        palette="Set2",
        title="UMAP: cell cycle phase",
    )
else:
    print("No valid S/G2M gene sets found for cell cycle scoring.")

## 8.5. Lineage scoring (MGE, CGE, LGE progenitors and neuroblasts)

We define marker sets for medial, caudal, and lateral ganglionic eminence (MGE, CGE, LGE) progenitors and neuroblasts. We compute per-cell scores for each lineage and visualize them on UMAP.

In [ ]:
marker_sets = {
    "MGE Progenitors":  ["Nkx2-1", "Sp9", "Prdm16", "Lhx6"],
    "MGE Neuroblasts":  ["Ackr3", "St18", "Maf", "Erbb4", "Sox6", "Foxp1"],
    "CGE Progenitors":  ["Nr2f2", "Pax6", "Sp8", "Ascl1"],
    "CGE Neuroblasts":  ["Prox1", "Synpr"],
    "LGE Progenitors":  ["Isl1", "Ebf1"],
    "LGE Neuroblasts":  ["Tac1", "Ddah1"],
}

# Score gene sets
for lineage_name, genes in marker_sets.items():
    gene_list = [g for g in genes if g in adata_gaba.var_names]
    if gene_list:
        sc.tl.score_genes(
            adata_gaba,
            gene_list=gene_list,
            score_name=f"{lineage_name}_score",
        )

sc.pl.umap(
    adata_gaba,
    color=[f"{ln}_score" for ln in marker_sets.keys()],
    cmap="viridis",
    ncols=3,
    title="Lineage scores on UMAP",
)

## 8.6. Assign lineage categories to clusters

For each cluster at the chosen Leiden resolution, we compute the mean lineage score and assign the dominant lineage. This yields a categorical `lineage_annotation` per cell.

In [ ]:
labels = adata_gaba.obs[key_for_umap].astype(str)
groups = sorted(labels.unique(), key=lambda x: int(x))

lineage_annotations = {}
for cluster in groups:
    mask = adata_gaba.obs[key_for_umap] == cluster
    scores = {
        ln: adata_gaba.obs.loc[mask, f"{ln}_score"].mean()
        for ln in marker_sets.keys()
        if f"{ln}_score" in adata_gaba.obs
    }
    best_lineage = max(scores, key=scores.get) if scores else "Unassigned"
    lineage_annotations[cluster] = best_lineage

adata_gaba.obs["lineage_annotation"] = labels.map(lineage_annotations)

sc.pl.umap(
    adata_gaba,
    color="lineage_annotation",
    title="UMAP: lineage annotation",
)

## 8.7. Differential expression and top markers per cluster

We restrict differential expression to a curated marker list, rank genes per cluster, and select one top marker per cluster. These markers are used to label clusters directly on UMAP.

In [ ]:
# From earlier: all marker genes present
all_markers = []
for genes in marker_sets.values():
    for g in genes:
        if g in adata_gaba.var_names:
            all_markers.append(g)
all_markers = list(dict.fromkeys(all_markers))

marker_list = [g for g in all_markers if g in adata_gaba.var_names]
adata_marker_subset = adata_gaba[:, marker_list].copy()

sc.tl.rank_genes_groups(
    adata_marker_subset,
    groupby=key_for_umap,
    method="wilcoxon",
    n_genes=20,
)

if hasattr(adata_marker_subset.obs[key_for_umap], "cat"):
    groups_de = adata_marker_subset.obs[key_for_umap].cat.categories
else:
    groups_de = np.unique(adata_marker_subset.obs[key_for_umap])

unique_top_genes = {}
used_genes = set()
for group in groups_de:
    gene_found = False
    for gene in adata_marker_subset.uns["rank_genes_groups"]["names"][group]:
        if gene not in used_genes:
            unique_top_genes[group] = gene
            used_genes.add(gene)
            gene_found = True
            break
    if not gene_found:
        unique_top_genes[group] = adata_marker_subset.uns["rank_genes_groups"]["names"][group][0]

print("Top marker genes per cluster:", unique_top_genes)

# UMAP with clusters and top markers as labels
umap = adata_gaba.obsm["X_umap"]
labels = adata_gaba.obs[key_for_umap].astype(str).values
groups = sorted(set(labels), key=lambda x: int(x))
palette = sns.color_palette("tab20", n_colors=len(groups))

plt.figure(figsize=(8, 7))
for i, group in enumerate(groups):
    idx = labels == group
    plt.scatter(umap[idx, 0], umap[idx, 1], s=10, color=palette[i], alpha=0.7)
    centroid = umap[idx].mean(axis=0)
    label_text = f"{group}\n{unique_top_genes.get(group, '')}"
    plt.text(
        centroid[0],
        centroid[1],
        label_text,
        fontsize=9,
        ha="center",
        va="center",
        bbox=dict(facecolor="white", alpha=0.8, boxstyle="round"),
    )

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("GABA UMAP with cluster labels and top markers")
plt.tight_layout()
plt.show()

## 8.8. Compositional summaries (genotype and batch)

We summarize the proportion of cells per cluster by genotype and by batch/animal, and visualize these as stacked bar plots. This highlights compositional shifts across Klf13 genotypes and potential batch imbalances.

In [ ]:
chosen_key = key_for_umap

# Genotype proportions (e.g. WT vs KO)
cluster_condition_counts = (
    adata_gaba.obs.groupby([chosen_key, "condition"]).size().unstack(fill_value=0)
)
cluster_condition_counts.index = cluster_condition_counts.index.astype(str)
cluster_order = sorted(cluster_condition_counts.index, key=lambda x: int(x))
cluster_condition_counts = cluster_condition_counts.loc[cluster_order]

cluster_condition_proportions = cluster_condition_counts.div(
    cluster_condition_counts.sum(axis=1), axis=0
)

cluster_condition_proportions = cluster_condition_proportions.rename(
    columns={
        "WT": "WT",
        "KO": r"$\it{Klf13}$ -/-",
        "Klf13 -/-": r"$\it{Klf13}$ -/-",
    }
)
colors = ["#9ecae1", "#fdd0a2"]  # light blue, light peach

fig, ax = plt.subplots(figsize=(8, 4.5))
cluster_condition_proportions.plot(
    kind="bar",
    stacked=True,
    width=0.75,
    color=colors,
    edgecolor="none",
    ax=ax,
)
ax.set_ylabel("Proportion of cells")
ax.set_xlabel("Cluster")
ax.set_title(r"WT vs $\it{Klf13}$ -/- proportions by cluster", pad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
ax.set_position([0.12, 0.2, 0.55, 0.65])
ax.legend(
    title="Condition",
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)
plt.show()

# Batch/animal composition
file_key = "batch"
cluster_file_counts = pd.crosstab(
    adata_gaba.obs[chosen_key],
    adata_gaba.obs[file_key],
)

cluster_file_counts.plot(
    kind="bar", stacked=True, figsize=(12, 6), colormap="tab20"
)
plt.ylabel("Number of cells")
plt.xlabel("Cluster")
plt.title("Cells from each batch/animal per GABA cluster")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Batch / animal", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8.9. Save GABA subset and annotations

Finally, we save the GABA-specific AnnData object containing:

- reclustered GABA cells,
- cell cycle and lineage scores,
- cluster-level lineage annotations,
- composition summaries (WT vs Klf13 KO, batch).

This object will be used for targeted differential expression and compositional analysis.

In [ ]:
adata_gaba.write("gaba_reclustered.h5ad")
print("GABA reclustering and annotation complete.")